In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
class PrimaryCapsules(nn.Module):
    def __init__(self, in_channels=256, num_capsules=32, capsule_dim=8, kernel_size=9, stride=2):
        super().__init__()
        self.num_capsules = num_capsules
        self.capsule_dim = capsule_dim
        self.conv = nn.Conv2d(in_channels, num_capsules * capsule_dim, kernel_size=kernel_size, stride=stride)

    @staticmethod
    def squash(x, dim=-1, eps=1e-8):
        sq_norm = (x ** 2).sum(dim=dim, keepdim=True)
        scale = sq_norm / (1.0 + sq_norm)
        return scale * x / torch.sqrt(sq_norm + eps)

    def forward(self, x):
        x = self.conv(x)  # (B, num_capsules*capsule_dim, H, W)
        bsz, _, h, w = x.shape
        x = x.view(bsz, self.num_capsules, self.capsule_dim, h, w)
        x = x.permute(0, 1, 3, 4, 2).contiguous().view(bsz, -1, self.capsule_dim)  # (B, N_primary, 8)
        return self.squash(x)


class DigitCapsules(nn.Module):
    def __init__(self, num_input_caps, input_dim=8, num_classes=4, output_dim=16, routing_iters=3):
        super().__init__()
        self.num_input_caps = num_input_caps
        self.num_classes = num_classes
        self.output_dim = output_dim
        self.routing_iters = routing_iters
        self.W = nn.Parameter(0.01 * torch.randn(1, num_input_caps, num_classes, output_dim, input_dim))

    @staticmethod
    def squash(x, dim=-1, eps=1e-8):
        sq_norm = (x ** 2).sum(dim=dim, keepdim=True)
        scale = sq_norm / (1.0 + sq_norm)
        return scale * x / torch.sqrt(sq_norm + eps)

    def forward(self, x):
        # x: (B, N_in, input_dim)
        bsz = x.size(0)
        x = x.unsqueeze(2).unsqueeze(-1)  # (B, N_in, 1, input_dim, 1)
        u_hat = torch.matmul(self.W.expand(bsz, -1, -1, -1, -1), x).squeeze(-1)  # (B, N_in, num_classes, output_dim)

        b = torch.zeros(bsz, self.num_input_caps, self.num_classes, device=x.device)

        for i in range(self.routing_iters):
            c = F.softmax(b, dim=-1)
            s = (c.unsqueeze(-1) * u_hat).sum(dim=1)
            v = self.squash(s)

            if i < self.routing_iters - 1:
                agreement = (u_hat * v.unsqueeze(1)).sum(dim=-1)
                b = b + agreement

        return v  # (B, num_classes, output_dim)

In [ ]:
class CapsNet(nn.Module):
    def __init__(self, num_classes=4, routing_iters=3, dropout_p=0.2):
        super().__init__()
        # Feature extractor to reduce spatial size before capsules
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 128, kernel_size=5, stride=2),  # 224 -> 110
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=5, stride=2),  # 110 -> 53
            nn.ReLU(inplace=True),
        )

        self.primary_caps = PrimaryCapsules(
            in_channels=256, num_capsules=32, capsule_dim=8, kernel_size=9, stride=2
        )
        self.dropout = nn.Dropout(p=dropout_p)

        # Dynamic capsule count (avoids hardcoded spatial assumptions)
        with torch.no_grad():
            dummy = torch.zeros(1, 3, 224, 224)
            dummy = self.conv1(dummy)
            dummy = self.primary_caps(dummy)
            num_primary_caps = dummy.size(1)

        self.digit_caps = DigitCapsules(
            num_input_caps=num_primary_caps,
            input_dim=8,
            num_classes=num_classes,
            output_dim=16,
            routing_iters=routing_iters,
        )

    def forward(self, x):
        x = self.conv1(x)
        x = self.primary_caps(x)
        x = self.dropout(x)
        digit_caps = self.digit_caps(x)  # (B, 4, 16)
        logits = torch.norm(digit_caps, dim=-1)  # (B, 4)
        return logits

In [ ]:
class CapsuleMarginLoss(nn.Module):
    def __init__(self, m_plus=0.9, m_minus=0.1, lambda_=0.5):
        super().__init__()
        self.m_plus = m_plus
        self.m_minus = m_minus
        self.lambda_ = lambda_

    def forward(self, logits, labels):
        # logits: (B, 4), labels: (B,)
        y = F.one_hot(labels, num_classes=logits.size(1)).float()
        left = F.relu(self.m_plus - logits).pow(2)
        right = F.relu(logits - self.m_minus).pow(2)
        loss = y * left + self.lambda_ * (1.0 - y) * right
        return loss.sum(dim=1).mean()

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CapsNet(num_classes=4, routing_iters=3, dropout_p=0.2).to(device)
criterion = CapsuleMarginLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

# Quick shape check: expected output is (B, 4)
images, labels = next(iter(train_loader))
images = images.to(device)
outputs = model(images)
print(outputs.shape)

In [ ]:
num_epochs = 25  # recommended range: 20-30

for epoch in range(num_epochs):
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)  # (B, 4)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        train_correct += (preds == labels).sum().item()
        train_total += labels.size(0)

    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)  # (B, 4)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    print(
        f"Epoch [{epoch + 1}/{num_epochs}] | "
        f"Train Loss: {train_loss / train_total:.4f} | Train Acc: {train_correct / train_total:.4f} | "
        f"Val Loss: {val_loss / val_total:.4f} | Val Acc: {val_correct / val_total:.4f}"
    )